In [ ]:
import pandas as pd
from sklearn.svm import OneClassSVM
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Verificar si hay valores nulos y eliminarlos de las columnas clave
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Función para entrenar One-Class SVM y graficar resultados
def train_and_plot_svm(data_column, column_name):
    # Entrenar el modelo One-Class SVM
    svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)  # Usando un kernel no lineal (RBF)
    data_column = data_column.values.reshape(-1, 1)  # Aseguramos que la entrada sea una columna vectorial
    svm.fit(data_column)

    # Predecir (1 para normal, -1 para anómalo)
    predictions = svm.predict(data_column)
    anomalies = data_column[predictions == -1]
    normal_points = data_column[predictions == 1]

    # Graficar resultados
    plt.figure(figsize=(10, 6))
    plt.plot(data_column, label=f'{column_name} (Valor real)', color='gray', alpha=0.7)
    plt.scatter(np.where(predictions == -1), anomalies, color='red', label='Anomalías', zorder=5)
    plt.scatter(np.where(predictions == 1), normal_points, color='green', label='Puntos normales', zorder=5)
    plt.title(f"One-Class SVM para {column_name}")
    plt.xlabel('Índice de Tiempo')
    plt.ylabel(column_name)
    plt.legend()
    plt.show()

# Entrenar y graficar el modelo One-Class SVM para cada una de las características
train_and_plot_svm(data_cleaned['VELOC X'], 'Velocidad X')
train_and_plot_svm(data_cleaned['VELOC Y'], 'Velocidad Y')
train_and_plot_svm(data_cleaned['VELOC Z'], 'Velocidad Z')
train_and_plot_svm(data_cleaned['ACEL X'], 'Aceleración X')
train_and_plot_svm(data_cleaned['ACEL Y'], 'Aceleración Y')
train_and_plot_svm(data_cleaned['ACEL Z'], 'Aceleración Z')
train_and_plot_svm(data_cleaned['TEMPERATURA'], 'Temperatura')


In [ ]:
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Verificar si hay valores nulos y eliminarlos de las columnas clave
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Función para entrenar One-Class SVM y proyectar en 2D usando t-SNE
def train_svm_and_tsne(data_cleaned, columns):
    # Seleccionamos solo las columnas relevantes
    X = data_cleaned[columns]

    # Entrenamos el modelo One-Class SVM
    svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)
    svm.fit(X)

    # Predecir (1 para normal, -1 para anómalo)
    predictions = svm.predict(X)

    # Proyectamos los datos a 2D usando t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X)

    # Visualizar el resultado
    plt.figure(figsize=(10, 8))
    plt.scatter(X_tsne[predictions == 1][:, 0], X_tsne[predictions == 1][:, 1], color='green', label='Normal', alpha=0.7)
    plt.scatter(X_tsne[predictions == -1][:, 0], X_tsne[predictions == -1][:, 1], color='red', label='Anomalía', alpha=0.7)
    plt.title(f"t-SNE y One-Class SVM para las columnas: {', '.join(columns)}")
    plt.xlabel('Componente 1')
    plt.ylabel('Componente 2')
    plt.legend()
    plt.show()

# Visualizamos los resultados para cada conjunto de características (velocidad, aceleración y temperatura)
train_svm_and_tsne(data_cleaned, ['VELOC X', 'VELOC Y', 'VELOC Z'])
train_svm_and_tsne(data_cleaned, ['ACEL X', 'ACEL Y', 'ACEL Z'])
train_svm_and_tsne(data_cleaned, ['TEMPERATURA'])


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Imprimir el Classification Report (asumiendo que se tienen etiquetas verdaderas)
# Nota: Se requiere una columna 'true_labels' con etiquetas reales para comparar.
if 'true_labels' in data_cleaned.columns:
    print(classification_report(data_cleaned['true_labels'], data_cleaned['outlier']))

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='TEMPERATURA',  # Colorear en función de la temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE para Velocidad, Aceleración y Temperatura',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'TEMPERATURA': 'Temperatura', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import plotly.express as px
from sklearn.metrics import classification_report, confusion_matrix

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Evaluación del modelo (Si hay etiquetas reales en 'ground_truth')
if 'ground_truth' in data_cleaned.columns:
    print("Classification Report:")
    print(classification_report(data_cleaned['ground_truth'], data_cleaned['outlier']))
    print("Confusion Matrix:")
    print(confusion_matrix(data_cleaned['ground_truth'], data_cleaned['outlier']))
else:
    print("No se encontraron etiquetas reales ('ground_truth'). Evaluación no disponible.")

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='TEMPERATURA',  # Colorear en función de la temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE para Velocidad, Aceleración y Temperatura',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'TEMPERATURA': 'Temperatura', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Crear una columna de color solo para los outliers
data_cleaned['color_temp'] = data_cleaned.apply(lambda row: row['TEMPERATURA'] if row['outlier'] == 1 else None, axis=1)

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='color_temp',  # Solo colorear los outliers por temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE para Velocidad, Aceleración y Temperatura (solo outliers coloreados)',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'color_temp': 'Temperatura (solo outliers)', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.15, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Imprimir el Classification Report (asumiendo que se tienen etiquetas verdaderas)
# Nota: Se requiere una columna 'true_labels' con etiquetas reales para comparar.
if 'true_labels' in data_cleaned.columns:
    print(classification_report(data_cleaned['true_labels'], data_cleaned['outlier']))

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='TEMPERATURA',  # Colorear en función de la temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE con mejor combinación para Velocidad, Aceleración y Temperatura',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'TEMPERATURA': 'Temperatura', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()